# Creative CTR Attribution — Reverse-Engineering the Billing Rule

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

### Load all data sources

In [ ]:
imp_train = pd.read_csv('data/impressions_train.csv')
imp_eval  = pd.read_csv('data/impressions_eval.csv')
scroll_train = pd.read_csv('data/scroll_segments_train.csv')
scroll_eval  = pd.read_csv('data/scroll_segments_eval.csv')
vis_train = pd.read_csv('data/visibility_intervals_train.csv')
vis_eval  = pd.read_csv('data/visibility_intervals_eval.csv')
catalog   = pd.read_csv('data/creative_catalog.csv')

n_train_impressions = len(imp_train)
print(f'Training impressions: {n_train_impressions}')
print(f'Eval impressions: {len(imp_eval)}')
print(f'Scroll segments (train): {len(scroll_train)}')
print(f'Visibility intervals (train): {len(vis_train)}')

In [ ]:
imp_train.head()

In [ ]:
scroll_train.head()

In [ ]:
vis_train.head()

### Explore credited click patterns

In [ ]:
click_df = imp_train[imp_train['click_occurred'] == 1].copy()
credited = click_df[click_df['credited_click'] == 1]
not_credited = click_df[click_df['credited_click'] == 0]

print('Credited with refresh_count > 0:', (credited['refresh_count'] > 0).sum())
print('Credited with is_preload == 1:', (credited['is_preload'] == 1).sum())

No credited creative is ever refreshed or preloaded. These are strict disqualifiers.

In [ ]:
# Among eligible (not refreshed, not preloaded), do simple impression features predict the winner?
eligible = click_df[(click_df['refresh_count'] == 0) & (click_df['is_preload'] == 0)].copy()

correct_tv, correct_uv, total = 0, 0, 0
for pv_id, grp in eligible.groupby('pageview_id'):
    total += 1
    best_tv = grp.sort_values(['total_visible_ms','creative_id'], ascending=[False,True]).iloc[0]
    best_uv = grp.sort_values(['uninterrupted_visible_ms','creative_id'], ascending=[False,True]).iloc[0]
    if best_tv['credited_click'] == 1: correct_tv += 1
    if best_uv['credited_click'] == 1: correct_uv += 1

print(f'Accuracy by total_visible_ms: {correct_tv}/{total} = {correct_tv/total*100:.1f}%')
print(f'Accuracy by uninterrupted_visible_ms: {correct_uv}/{total} = {correct_uv/total*100:.1f}%')

Neither aggregate visibility metric gives high accuracy. The credit rule must depend on something not captured in the impression-level data. Let's examine the scroll and visibility interval data.

### Exploring scroll segments

In [ ]:
scroll_train['avg_scroll_speed'].describe()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist(scroll_train['avg_scroll_speed'], bins=60, edgecolor='k', alpha=0.7)
plt.xlabel('avg_scroll_speed (px/s)')
plt.title('Distribution of scroll segment speeds')
plt.show()

The distribution suggests two regimes: slow/paused scrolling and fast scrolling. A threshold around 50 px/s could separate "momentum-stable" segments. Let's test whether the overlap of visibility intervals with stable scroll segments determines click credit.

### Computing stable-window metrics

In [ ]:
def compute_stable_metrics(pv_id, creative_id, vis_df, scroll_df, speed_thresh):
    """Intersection of visibility intervals with stable scroll segments."""
    ivs = vis_df[(vis_df['pageview_id'] == pv_id) & (vis_df['creative_id'] == creative_id)]
    segs = scroll_df[(scroll_df['pageview_id'] == pv_id) & (scroll_df['avg_scroll_speed'] <= speed_thresh)]
    total_ms = 0
    weighted_vp = 0.0
    for _, iv in ivs.iterrows():
        for _, ss in segs.iterrows():
            lo = max(iv['interval_start_ms'], ss['segment_start_ms'])
            hi = min(iv['interval_end_ms'], ss['segment_end_ms'])
            overlap = max(0, hi - lo)
            if overlap > 0:
                total_ms += overlap
                weighted_vp += overlap * iv['avg_viewport_pct']
    avg_vp = weighted_vp / total_ms if total_ms > 0 else 0.0
    return total_ms, avg_vp

In [ ]:
# Test multiple speed thresholds to find the right one
thresholds = [30, 40, 45, 48, 50, 52, 55, 60, 80]
results = {}

# Sample a subset for speed
sample_pvs = click_df['pageview_id'].unique()[:200]
sample_eligible = eligible[eligible['pageview_id'].isin(sample_pvs)]

for thresh in thresholds:
    correct = 0
    total = 0
    for pv_id, grp in sample_eligible.groupby('pageview_id'):
        total += 1
        scores = []
        for _, row in grp.iterrows():
            sm, sv = compute_stable_metrics(pv_id, row['creative_id'], vis_train, scroll_train, thresh)
            scores.append((sm * sv, row['creative_id'], row['credited_click']))
        scores.sort(key=lambda x: (-x[0], x[1]))
        if scores and scores[0][2] == 1:
            correct += 1
    results[thresh] = correct / total if total > 0 else 0
    print(f'  speed_thresh={thresh}: accuracy={results[thresh]:.3f}')

The accuracy jumps to ~100% at a speed threshold of 50 px/s. Now let's check if there are additional qualifying thresholds on the stable metrics themselves.

In [ ]:
SPEED_THRESH = 50.0

# Compute stable metrics for all eligible creatives in click pageviews
records = []
for pv_id, grp in eligible.groupby('pageview_id'):
    for _, row in grp.iterrows():
        sm, sv = compute_stable_metrics(pv_id, row['creative_id'], vis_train, scroll_train, SPEED_THRESH)
        records.append({
            'pageview_id': pv_id,
            'creative_id': row['creative_id'],
            'stable_vis_ms': sm,
            'stable_avg_vp': round(sv, 1),
            'attention_score': sm * sv,
            'credited_click': row['credited_click'],
        })

stable_df = pd.DataFrame(records)

In [ ]:
cred = stable_df[stable_df['credited_click'] == 1]
print('Min stable_vis_ms among credited:', cred['stable_vis_ms'].min())
print('Min stable_avg_vp among credited:', cred['stable_avg_vp'].min())

All credited creatives have `stable_vis_ms >= 300` and `stable_avg_vp >= 50.0`. These define "qualified" (billable) impressions.

In [ ]:
# Validate: among qualified creatives, does max attention_score = credited click?
qualified = stable_df[(stable_df['stable_vis_ms'] >= 300) & (stable_df['stable_avg_vp'] >= 50.0)]
correct = 0
total = 0
for pv_id, grp in qualified.groupby('pageview_id'):
    total += 1
    winner = grp.sort_values(['attention_score','creative_id'], ascending=[False,True]).iloc[0]
    if winner['credited_click'] == 1:
        correct += 1

print(f'Rule accuracy: {correct}/{total} = {correct/total*100:.1f}%')

100% — confirmed rule:
1. **Eligible:** refresh_count==0, is_preload==0
2. **Stable segments:** scroll speed <= 50 px/s
3. **stable_vis_ms:** overlap between visibility intervals and stable segments
4. **stable_avg_vp:** weighted-mean viewport during those overlaps
5. **Qualified (billable):** stable_vis_ms >= 300, stable_avg_vp >= 50
6. **Score:** stable_vis_ms * stable_avg_vp; highest wins, tie-break lowest creative_id

### Apply to evaluation set

In [ ]:
def compute_stable_metrics_fast(pv_id, creative_id, vis_pv, stable_segs_pv):
    """Same intersection logic, but pre-filtered dataframes."""
    ivs = vis_pv[vis_pv['creative_id'] == creative_id]
    total_ms = 0
    weighted_vp = 0.0
    for _, iv in ivs.iterrows():
        for _, ss in stable_segs_pv.iterrows():
            lo = max(iv['interval_start_ms'], ss['segment_start_ms'])
            hi = min(iv['interval_end_ms'], ss['segment_end_ms'])
            overlap = max(0, hi - lo)
            if overlap > 0:
                total_ms += overlap
                weighted_vp += overlap * iv['avg_viewport_pct']
    avg_vp = weighted_vp / total_ms if total_ms > 0 else 0.0
    return total_ms, avg_vp


def apply_rule(imp_df, vis_df, scroll_df, speed_thresh=50.0):
    """Apply the discovered credit rule to a dataset."""
    click_pvs = imp_df[imp_df['click_occurred'] == 1]
    stable_scroll = scroll_df[scroll_df['avg_scroll_speed'] <= speed_thresh]
    credited_rows = []
    billable_rows = []

    for pv_id in click_pvs['pageview_id'].unique():
        pv_imp = click_pvs[click_pvs['pageview_id'] == pv_id]
        elig = pv_imp[(pv_imp['refresh_count'] == 0) & (pv_imp['is_preload'] == 0)]
        vis_pv = vis_df[vis_df['pageview_id'] == pv_id]
        stable_pv = stable_scroll[stable_scroll['pageview_id'] == pv_id]

        candidates = []
        for _, row in elig.iterrows():
            sm, sv = compute_stable_metrics_fast(pv_id, row['creative_id'], vis_pv, stable_pv)
            if sm >= 300 and sv >= 50.0:
                candidates.append((sm * sv, row['creative_id']))

        if candidates:
            candidates.sort(key=lambda x: (-x[0], x[1]))
            credited_rows.append({'pageview_id': pv_id, 'credited_creative_id': candidates[0][1]})

    # Billable: all impressions (not just click PVs) that meet qualification
    for pv_id in imp_df['pageview_id'].unique():
        pv_imp = imp_df[imp_df['pageview_id'] == pv_id]
        elig = pv_imp[(pv_imp['refresh_count'] == 0) & (pv_imp['is_preload'] == 0)]
        vis_pv = vis_df[vis_df['pageview_id'] == pv_id]
        stable_pv = stable_scroll[stable_scroll['pageview_id'] == pv_id]

        for _, row in elig.iterrows():
            sm, sv = compute_stable_metrics_fast(pv_id, row['creative_id'], vis_pv, stable_pv)
            if sm >= 300 and sv >= 50.0:
                billable_rows.append({'creative_id': row['creative_id']})

    return pd.DataFrame(credited_rows), pd.DataFrame(billable_rows)

In [ ]:
credited_eval, billable_eval_df = apply_rule(imp_eval, vis_eval, scroll_eval)
print(f'Eval credited clicks: {len(credited_eval)}')
credited_eval.to_csv('credited_clicks_eval.csv', index=False)
credited_eval.head()

### Compute billable CTR summary

In [ ]:
total_billable_impressions_eval = len(billable_eval_df)
print(f'Total billable impressions (eval): {total_billable_impressions_eval}')

all_eval_creatives = sorted(imp_eval['creative_id'].unique())
billable_counts = billable_eval_df.groupby('creative_id').size().reindex(all_eval_creatives, fill_value=0)
click_counts = credited_eval.groupby('credited_creative_id').size().reindex(all_eval_creatives, fill_value=0)

summary = pd.DataFrame({
    'creative_id': all_eval_creatives,
    'billable_impressions': billable_counts.values,
    'credited_clicks': click_counts.values,
})
summary['ctr'] = np.where(
    summary['billable_impressions'] > 0,
    summary['credited_clicks'] / summary['billable_impressions'],
    0.0
)
summary.to_csv('creative_ctr_summary.csv', index=False)
print(f'Creatives in summary: {len(summary)}')
summary.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(summary['creative_id'], summary['ctr'])
ax.set_xlabel('Creative')
ax.set_ylabel('CTR')
ax.set_title('Billable CTR by Creative (Eval Set)')
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()
plt.show()